# Sprint 2 Live — Secure-service workshops

Use this one Live notebook for LS05–LS08. Work in a separate Live copy; keep Campus checkpoint and assessment work in the Campus notebook.

Missing or wrong caller keys and limited requests stop before generation. Each admitted supported request uses the actual OpenRouter service path and may consume provider quota.

**Colab runtime:** `ServiceProcess` runs Uvicorn in a managed child process bound to loopback, checks readiness with a timeout, and provides an explicit stop. Do not run `uvicorn.run()` in a cell or expose the service through ngrok; a foreground server call would prevent that cell from returning. HTTP request cells wait for their bounded client timeout while leaving the notebook runtime available for subsequent cells.


## Instructor guides and live decks in Notion

- [LS 5 — API key auth and secrets management](https://app.notion.com/p/3ea5a55972ad815a8aacc8b5e81a2e48?pvs=204)
- [LS 6 — Rate limiting AI endpoints](https://app.notion.com/p/3ea5a55972ad81a4ac37e3e6d49b1933?pvs=204)
- [LS 7 — Secrets safety in practice](https://app.notion.com/p/3ea5a55972ad816f954ff56542286d89?pvs=204)
- [LS 8 — Sprint 2 checkpoint](https://app.notion.com/p/3ea5a55972ad81f89fdfc35996bca7f1?pvs=204)


## 1. Set up once

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
import sys
from zipfile import ZipFile

REPOSITORY = "https://github.com/richhiey/ai-app-dev_Mod-B.git"
REPO = Path("/content/ai-app-dev_Mod-B")
PROJECT = Path("/content/fieldcare-sprint-2-live")
CAMPUS_CHECKPOINT = ""  # Optional uploaded Sprint 1 Campus source checkpoint.

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(REPO)], check=True)
else:
    # Colab runtimes persist /content; refresh an existing clone before reinstalling it.
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-c", str(REPO / "requirements.lock")], check=True)
sys.path.insert(0, str(REPO / "src"))  # Make the editable package visible to this running kernel.

from module_b.openrouter import DEFAULT_CHAT_MODEL, require_openrouter_key
require_openrouter_key(prompt=True)
os.environ.setdefault("OPENROUTER_MODEL", DEFAULT_CHAT_MODEL)

if CAMPUS_CHECKPOINT:
    PROJECT.mkdir(parents=True, exist_ok=True)
    with ZipFile(CAMPUS_CHECKPOINT) as archive:
        archive.extractall(PROJECT)
elif not PROJECT.exists():
    shutil.copytree(REPO / "examples" / "fieldcare", PROJECT)
print("Module B source:", REPO)
print("Sprint 2 workspace:", PROJECT)

In [ ]:
import os
import secrets
import httpx
from module_b.runtime import ServiceProcess

CALLERS = {
    "dispatch": "FIELDCARE_DISPATCH_KEY",
    "partner": "FIELDCARE_PARTNER_KEY",
}
caller_keys = {label: secrets.token_urlsafe(32) for label in CALLERS}
for label, variable in CALLERS.items():
    os.environ[variable] = caller_keys[label]
print("Temporary caller keys are loaded in memory; their values are not displayed.")

## LS05 — Authenticate before application work

Attach the supplied middleware after route registration. Predict which requests reach the handler, then compare the actual HTTP statuses. Keep the caller key and OpenRouter key separate.

In [ ]:
settings = """from module_b.security import LimitPolicy

CALLER_ENV = {
    "dispatch": "FIELDCARE_DISPATCH_KEY",
    "partner": "FIELDCARE_PARTNER_KEY",
}
POLICY = None
"""
(PROJECT / "app" / "security_settings.py").write_text(settings)

main_path = PROJECT / "app" / "main.py"
main_source = main_path.read_text()
registration = """
from module_b.security import SecurityMiddleware, protected_post_paths
from .security_settings import CALLER_ENV, POLICY
app.add_middleware(
    SecurityMiddleware,
    caller_env=CALLER_ENV,
    protected_paths=protected_post_paths(app),
    policy=POLICY,
)
"""
# Replace any registration added by an earlier run, including the legacy
# unmarked block, so rerunning this cell repairs the persisted workspace.
legacy_registration = "\nfrom module_b.security import SecurityMiddleware, protected_post_paths\n"
if legacy_registration in main_source:
    main_source = main_source.split(legacy_registration, 1)[0]
generated_marker = "\n# BEGIN COLAB SECURITY MIDDLEWARE"
if generated_marker in main_source:
    main_source = main_source.split(generated_marker, 1)[0]
registration = registration.replace(
    "\nfrom module_b.security import",
    "\n# BEGIN COLAB SECURITY MIDDLEWARE\nfrom module_b.security import",
)
registration = registration.rstrip() + "\n# END COLAB SECURITY MIDDLEWARE\n"
main_path.write_text(main_source.rstrip() + registration)

def start_service():
    service = ServiceProcess(
        project_dir=PROJECT,
        env={variable: caller_keys[label] for label, variable in CALLERS.items()},
    )
    service.start()
    return service

service = start_service()

In [ ]:
payload = {
    "question": "Which filter and airflow checks are documented?",
    "equipment_id": "EQ-FC-1002",
}
with httpx.Client(base_url=service.base_url, timeout=90) as client:
    cases = [
        ("missing caller key", client.post("/v1/diagnose", json=payload)),
        ("wrong caller key", client.post(
            "/v1/diagnose", json=payload, headers={"X-API-Key": "not-a-valid-key"}
        )),
        ("recognized caller", client.post(
            "/v1/diagnose", json=payload,
            headers={"X-API-Key": caller_keys["dispatch"]}
        )),
        ("recognized caller, invalid body", client.post(
            "/v1/diagnose", json={},
            headers={"X-API-Key": caller_keys["dispatch"]}
        )),
    ]

for label, response in cases:
    body = response.json()
    print(label, "->", response.status_code, body.get("status", body.get("detail")))
service.stop()

## LS06 — Make per-caller limiting visible

The configured window counts admitted requests by caller across routes. This Live exercise uses a one-request, 60-second window. The admitted requests below are real model calls; the exhausted request is rejected before retrieval and generation.

In [ ]:
# Keep the same routes and caller mapping; change only the policy.
settings = settings.replace("POLICY = None", "POLICY = LimitPolicy(allowance=1, window_seconds=60)")
(PROJECT / "app" / "security_settings.py").write_text(settings)
service = start_service()

In [ ]:
with httpx.Client(base_url=service.base_url, timeout=90) as client:
    first = client.post(
        "/v1/diagnose", json=payload,
        headers={"X-API-Key": caller_keys["dispatch"]}
    )
    limited = client.post(
        "/v1/diagnose", json=payload,
        headers={"X-API-Key": caller_keys["dispatch"]}
    )
    partner = client.post(
        "/v1/diagnose", json=payload,
        headers={"X-API-Key": caller_keys["partner"]}
    )

for label, response in [
    ("first Dispatch request", first),
    ("Dispatch over limit", limited),
    ("Partner has an independent allowance", partner),
]:
    body = response.json()
    print(label, "->", response.status_code,
          body.get("status", body.get("detail")),
          "Retry-After:", response.headers.get("retry-after"))
service.stop()

## LS07 — Follow a synthetic marker through three local sinks

This exercise makes no provider call and sends no HTTP request. Use only the visibly synthetic marker. A disposable local folder holds an intentionally exposed log policy, prompt policy, and Git history. Run the before check, apply each prevention, then repeat the workflow with a fresh `RUN_NAME`.


In [ ]:
from module_b.secret_workshop import prepare_secret_workshop

RUN_NAME = "attempt_1"  # Change this to create a separate fresh fixture.
MARKER = f"SYNTHETIC_{RUN_NAME.upper()}_NOT_A_CREDENTIAL"
FIXTURE = prepare_secret_workshop(
    Path("/content/module-b-ls07") / RUN_NAME,
    marker=MARKER,
    repo_root=REPO,
)
checks = {}
print("Disposable local fixture:", FIXTURE)
print("Provider calls:", 0, "| HTTP requests:", 0)


### Before prevention — inspect the actual local artifacts

The starter `log_policy.py` and `prompt_policy.py` deliberately pass too much data. Run their real functions with synthetic values. The local exception below is only a Python input to the logger; it is not an HTTP or provider failure.


In [ ]:
import json
import runpy

event = {
    "request_id": "synthetic-local-attempt-01",
    "route": "/v1/diagnose",
    "outcome": "completed",
    "headers": {"X-API-Key": MARKER},
}
log_record = runpy.run_path(str(FIXTURE / "log_policy.py"))["log_record"]
ordinary = log_record(event)
failed = log_record(
    {**event, "request_id": "synthetic-local-attempt-02", "outcome": "failed"},
    ValueError("Synthetic local exception contains " + MARKER),
)
LOG_PATH = FIXTURE / "observed.log"
LOG_PATH.write_text(json.dumps([ordinary, failed], indent=2) + "\n")

build_prompt = runpy.run_path(str(FIXTURE / "prompt_policy.py"))["build_prompt"]
payload = build_prompt(
    "Which filter check should the technician make?",
    "Synthetic HX guidance: inspect the intake filter.",
    {"provider_key": MARKER},
)
PAYLOAD_PATH = FIXTURE / "outbound-payload.json"
PAYLOAD_PATH.write_text(json.dumps(payload, indent=2) + "\n")

log_text = LOG_PATH.read_text()
payload_text = PAYLOAD_PATH.read_text()
print("Marker in saved ordinary/error logs:", MARKER in log_text)
print("Marker in locally built prompt payload:", MARKER in payload_text)
print("Saved ordinary/error log:")
display(json.loads(log_text))
print("Locally assembled outbound payload:")
display(json.loads(payload_text))


### Prevent disclosure in logs

Open `examples/patterns/secret_workshop/log_safe.py` and adapt its field selection in `FIXTURE/log_policy.py` using Colab’s file editor. Keep the request ID, route, outcome, and a bounded error code; drop headers and exception text. Then rerun the next cell and inspect `observed.log`.


In [ ]:
log_record = runpy.run_path(str(FIXTURE / "log_policy.py"))["log_record"]
ordinary = log_record(event)
failed = log_record(
    {**event, "request_id": "synthetic-local-attempt-02", "outcome": "failed"},
    ValueError("Synthetic local exception contains " + MARKER),
)
LOG_PATH.write_text(json.dumps([ordinary, failed], indent=2) + "\n")
log_text = LOG_PATH.read_text()
checks["ordinary and error logs exclude the marker"] = MARKER not in log_text
checks["safe diagnostic fields remain"] = all(
    row.get("request_id") and row.get("route") == "/v1/diagnose"
    and row.get("outcome") in {"completed", "failed"}
    for row in json.loads(log_text)
)
print("Marker in saved logs:", MARKER in log_text)
print("Retained record fields:", sorted(ordinary))


### Keep runtime configuration out of the prompt

Adapt `examples/patterns/secret_workshop/prompt_safe.py` in `FIXTURE/prompt_policy.py`. The prompt needs the question and task context, not runtime configuration. No request is sent to a model.


In [ ]:
build_prompt = runpy.run_path(str(FIXTURE / "prompt_policy.py"))["build_prompt"]
payload = build_prompt(
    "Which filter check should the technician make?",
    "Synthetic HX guidance: inspect the intake filter.",
    {"provider_key": MARKER},
)
PAYLOAD_PATH.write_text(json.dumps(payload, indent=2) + "\n")
payload_text = PAYLOAD_PATH.read_text()
checks["outbound prompt excludes the marker"] = MARKER not in payload_text
checks["question and useful context remain"] = (
    "Which filter check" in payload_text and "inspect the intake filter" in payload_text
)
print("Marker in outbound payload:", MARKER in payload_text)
print("Question and context retained:", checks["question and useful context remain"])


### Stop future Git exposure; keep history distinct

This disposable repository has a synthetic `.env` in its initial commit and a new `local.key`. Add both patterns to `.gitignore`, then use the visible `git rm --cached` command to remove `.env` from the current index while leaving the local file in place. This does not erase the initial commit.


In [ ]:
(FIXTURE / ".gitignore").write_text(".env\nlocal.key\n")
subprocess.run(
    ["git", "-C", str(FIXTURE), "rm", "--cached", ".env"],
    check=True, capture_output=True, text=True,
)
tracked = subprocess.run(
    ["git", "-C", str(FIXTURE), "ls-files", "--error-unmatch", ".env"],
    capture_output=True, text=True,
).returncode == 0
ignored_env = subprocess.run(
    ["git", "-C", str(FIXTURE), "check-ignore", "-q", ".env"]
).returncode == 0
ignored_key = subprocess.run(
    ["git", "-C", str(FIXTURE), "check-ignore", "-q", "local.key"]
).returncode == 0
initial_env = subprocess.run(
    ["git", "-C", str(FIXTURE), "show", "HEAD:.env"],
    check=True, capture_output=True, text=True,
).stdout
checks.update({
    "new .env is ignored": ignored_env,
    "new local.key is ignored": ignored_key,
    ".env is no longer tracked": not tracked,
    "initial commit still contains the marker": MARKER in initial_env,
})
print(".env ignored / tracked:", ignored_env, tracked)
print("local.key ignored:", ignored_key)
print("Marker remains in initial commit:", MARKER in initial_env)


### Record a fresh-marker check

Change `RUN_NAME` in the fixture cell to a new value and repeat the before check, both code edits, and the Git prevention. Then run the summary below. It checks the artifacts created in that fresh workspace; it does not claim to scan your actual project.


In [ ]:
for name, passed in checks.items():
    print(("PASS" if passed else "REVIEW"), name)
print("Evidence kind: local synthetic fixture")
print("Provider calls:", 0, "| HTTP requests:", 0)


## LS08 — Checkpoint and explain

In your notes, identify which requests reached OpenRouter, which stopped earlier, where each credential belongs, and why the in-memory limiter is not a distributed quota. Export the app and data; save the Live notebook separately.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

CHECKPOINT_OUT = Path("/content/sprint-2-live-checkpoint.zip")
with ZipFile(CHECKPOINT_OUT, "w", ZIP_DEFLATED) as archive:
    for folder in ("app", "data"):
        for path in (PROJECT / folder).rglob("*"):
            if path.is_file() and "__pycache__" not in path.parts and path.suffix != ".pyc":
                archive.write(path, path.relative_to(PROJECT))
print("Saved checkpoint:", CHECKPOINT_OUT)

### Carry forward

Keep this Live workspace separate from Campus. For Sprint 3, import the checkpoint you intend to extend and document any changes you apply back to your project.